# 🗂️ Notebook 2 — Stock Exchange: Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/stock-exchange
uv sync
```

Then **select the `.venv` kernel** in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

This lab uses only the Python standard library + `pydantic` — nothing else to install, no Docker required.


## The three core entities

| Entity | What it represents                                                        |
|--------|---------------------------------------------------------------------------|
| Order  | A trader's intent to buy or sell N shares at price P.                     |
| Trade  | The result of two orders matching — two traders end up with new positions. |
| Book   | Per-symbol, all resting (unmatched) orders, sorted by price then time.    |

Everything else — users, positions, market data — is downstream of these three.

## Bad → Best #1: never use `float` for money

The single most common beginner bug in finance code: using `float`. Binary floating point can't represent `0.1` exactly, so sums drift. Over millions of orders that drift becomes real money missing from real accounts.

**Fix**: store prices as **integer ticks** (smallest allowed price increment, e.g. $0.01 → 1 tick). Convert to dollars only at the UI boundary.

In [ ]:
# BAD: floats for money -----------------------------------------------
# Every schoolkid knows 0.1 + 0.2 == 0.3 … every computer disagrees:
print(f"BAD  0.1 + 0.2 = {0.1 + 0.2!r}  (not 0.3 !)")

# Accumulate one penny at a time, the way a naive P&L calculation would:
total = 0.0
for _ in range(1_000):
    total += 0.07              # a 7-cent fee
print(f"BAD  1,000 x $0.07 = {total!r}   (expected 70.0)")
print(f"     off by ${abs(70.0 - total):.2e} — looks tiny, until you scale to")
print(f"     millions of trades across millions of accounts.")

# BEST: integers of ticks (cents) -------------------------------------
TICK = 1  # 1 tick == 1 cent
best_total_ticks = 0
for _ in range(1_000):
    best_total_ticks += 7      # 7 ticks == $0.07
print(f"\nBEST 1,000 x 7 ticks = {best_total_ticks} ticks = ${best_total_ticks/100:,.2f}")
assert best_total_ticks == 7_000    # exact — no drift, ever

## Bad → Best #2: validate at the boundary

The gateway is an untrusted edge. If we let *any* dict through, bad inputs poison the matcher (`qty = -5` would let a trader *mint* shares). Use `pydantic` to reject bad data **before** it reaches the hot path.

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel, Field, PositiveInt, ValidationError

Side     = Literal["buy", "sell"]
OrderType = Literal["limit", "market", "ioc", "fok"]
#                     ^       ^        ^      ^
#                     |       |        |      └ fill-or-kill: all-or-nothing, else cancel
#                     |       |        └────── immediate-or-cancel: fill what you can, cancel rest
#                     |       └─────────────── take any price available right now
#                     └─────────────────────── rest on book at exactly this price or better

class NewOrder(BaseModel):
    client_order_id: str = Field(min_length=1, max_length=64)   # idempotency key
    symbol: str = Field(pattern=r"^[A-Z]{1,5}$")
    side: Side
    type: OrderType = "limit"
    price_ticks: Optional[PositiveInt] = None   # required for limit / ioc / fok
    qty: PositiveInt
    trader: str

    def model_post_init(self, __context):
        if self.type != "market" and self.price_ticks is None:
            raise ValueError("non-market orders must specify price_ticks")

class Trade(BaseModel):
    seq: PositiveInt              # global monotonic sequence # (set by sequencer)
    symbol: str
    buy_order_id:  str
    sell_order_id: str
    price_ticks: PositiveInt
    qty: PositiveInt

good = NewOrder(client_order_id="c-1", symbol="AAPL", side="buy",
                price_ticks=19050, qty=10, trader="alice")
print("✅", good)

for bad in [
    dict(client_order_id="c-2", symbol="aapl",  side="buy",  price_ticks=1, qty=1, trader="bob"),     # lowercase symbol
    dict(client_order_id="c-3", symbol="AAPL", side="buy",  price_ticks=0, qty=1, trader="bob"),     # non-positive price
    dict(client_order_id="c-4", symbol="AAPL", side="buy",  price_ticks=1, qty=-5, trader="bob"),    # negative qty
    dict(client_order_id="c-5", symbol="AAPL", side="sideways", price_ticks=1, qty=1, trader="bob"), # bad side
    dict(client_order_id="c-6", symbol="AAPL", side="buy", qty=1, trader="bob"),                    # limit without price
]:
    try:
        NewOrder(**bad)
    except ValidationError as e:
        print("❌ rejected:", bad.get("client_order_id"), "→", e.errors()[0]["msg"])
    except ValueError as e:
        print("❌ rejected:", bad.get("client_order_id"), "→", e)

## Bad → Best #3: idempotency via `client_order_id`

The network drops packets. A trader retries. Without an idempotency key you place the order **twice**. Always require a unique `client_order_id` and reject (or re-ack the original) on duplicates.

In [ ]:
# Tiny idempotent submit() — reject duplicate client_order_id
seen = {}                                     # client_order_id → server order id
next_server_id = iter(range(1, 10**9)).__next__

def submit(req: NewOrder):
    if req.client_order_id in seen:
        return {"status": "duplicate", "order_id": seen[req.client_order_id]}
    sid = next_server_id()
    seen[req.client_order_id] = sid
    return {"status": "accepted", "order_id": sid}

r1 = submit(good)
r2 = submit(good)     # client retries the exact same request
print(r1); print(r2)
assert r1["order_id"] == r2["order_id"]   # same order, not two!

## Public API surface

A real exchange speaks **FIX** (a 1990s pipe-delimited protocol that every bank already supports) and/or a binary format for speed. For a learning exchange we expose a tiny HTTP + WebSocket API:

| Method | Path                   | Purpose                                        |
|--------|------------------------|------------------------------------------------|
| POST   | `/orders`              | Submit a `NewOrder` (see the model above)      |
| DELETE | `/orders/{id}`         | Cancel a resting order                         |
| GET    | `/book/{symbol}?depth=10` | Snapshot of top-N price levels              |
| WS     | `/feed/{symbol}`       | Live stream of trades + top-of-book updates    |

On the wire it's just JSON — so a `NewOrder` from a trader looks like:

In [ ]:
print(good.model_dump_json(indent=2))

# …and we can round-trip it back from what arrived on the gateway socket:
on_wire = good.model_dump_json()
parsed  = NewOrder.model_validate_json(on_wire)
assert parsed == good
print("\n✅ wire → model round-trip OK")

## Takeaways

- Model money as **integer ticks**. Floats cost real dollars in real systems.
- Validate at the gateway with `pydantic`. The matcher trusts its input.
- Every order carries a `client_order_id` — the single best defence against duplicate submissions.
- Support at least `limit`, `market`, `ioc`, `fok` from day one; they're easy to add now and very hard to bolt on later.

Next: **Notebook 3** — we build the matching engine itself.